# Atividade 001 — Pipeline de Classificação: Penguins

**Disciplina:** Inteligência Artificial  
**Professor:** Marcelo Batista  

---

## Objetivo
Treinar um modelo de ML no dataset `penguins` de forma **reprodutível** usando:
- **Dados → Treino → Teste**
- **Pipeline + ColumnTransformer** (tratando missing + categorias)
- **Baseline:** Logistic Regression
- **Melhoria simples:** Random Forest
- **Avaliação:** accuracy, classification_report, matriz de confusão
- **Interpretação humana** dos erros (confusões entre classes)

---

## Parte 1 — Definição do problema

### Contrato do projeto

1. **Dataset escolhido:** `penguins` (Seaborn)
2. **Target (y):** `species` - espécie do pinguim (`Adelie`, `Chinstrap`, `Gentoo`)
3. **Tipo do problema:** Classificação **multiclasse** (3 classes)
4. **Features (X) utilizadas:**
   - Numéricas: `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`
   - Categóricas: `island`, `sex`
5. **Colunas removidas:** nenhuma, todas as colunas disponíveis têm potencial informativo para distinguir a espécie.

> **Justificativa do dataset:** O `penguins` é o sucessor moderno do `iris`. Tem variáveis categóricas reais (`island`, `sex`) e missing values, tornando o pipeline mais completo e próximo de um cenário real.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

---

## Parte 2 — Diagnóstico dos dados

In [2]:
# Carregar o dataset penguins
df = sns.load_dataset("penguins")

print("Shape:", df.shape)

print("\nTipos das colunas:")
print(df.dtypes)

print("\nFaltantes por coluna:")
print(df.isna().sum().sort_values(ascending=False))

print("\nDistribuição do target (species):")
print(df["species"].value_counts())

print("\nProporção do target:")
print(df["species"].value_counts(normalize=True).rename("proportion"))

Shape: (344, 7)

Tipos das colunas:
species               object
island                object
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                   object
dtype: object

Faltantes por coluna:
sex                  11
bill_depth_mm         2
bill_length_mm        2
flipper_length_mm     2
body_mass_g           2
island                0
species               0
dtype: int64

Distribuição do target (species):
species
Adelie       152
Gentoo       124
Chinstrap     68
Name: count, dtype: int64

Proporção do target:
species
Adelie       0.441860
Gentoo       0.360465
Chinstrap    0.197674
Name: proportion, dtype: float64


In [3]:
display(df.head())

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


### Observações do diagnóstico

- **344 linhas × 7 colunas**
- **4 colunas numéricas** (`bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`) e **2 categóricas** (`island`, `sex`) além do target
- **Missing values:**
  - `sex`: 17 faltantes (~5%) → necessita imputação pela moda
  - Numéricas: 3 faltantes cada (~0.9%) → imputação pela mediana
- **Distribuição do target:** dataset levemente desbalanceado — `Adelie` é maioria (44%), `Chinstrap` é minoria (20%)
- **Conclusão do diagnóstico:** precisamos de um pipeline com imputação (missing) e codificação (categóricas)

---

## Parte 3 — Construção do Pipeline

In [4]:
# Definir features e target
TARGET = "species"
NUM_COLS = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
CAT_COLS = ["island", "sex"]

X = df.drop(columns=[TARGET])
y = df[TARGET]

# Separar treino e teste com estratificação (mantém proporção de classes)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Tamanho treino:", X_train.shape)
print("Tamanho teste: ", X_test.shape)

Tamanho treino: (275, 6)
Tamanho teste:  (69, 6)


In [5]:
# Pipeline numérico: imputar com mediana + padronizar
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Pipeline categórico: imputar com moda + one-hot encoding
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Pré-processador combinado
preprocess = ColumnTransformer([
    ("num", num_pipe, NUM_COLS),
    ("cat", cat_pipe, CAT_COLS)
])

### Função auxiliar de avaliação

In [6]:
LABELS = ["Adelie", "Chinstrap", "Gentoo"]

def avaliar_classificacao(nome, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    print(f"\n=== {nome} ===")
    print(f"Accuracy: {acc:.4f}")
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, target_names=LABELS))
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    print("Matriz de confusão (linhas=real, colunas=previsto):")
    print(f"{'':>12}", "  ".join(f"{l:>10}" for l in LABELS))
    for i, label in enumerate(LABELS):
        print(f"{label:>12}", "  ".join(f"{v:>10}" for v in cm[i]))
    return {"acc": acc, "cm": cm}

---

## 🤖 Parte 4 — Modelos: Baseline + Melhoria

### Baseline — Logistic Regression

Modelo **simples e interpretável**. Assume relações lineares entre features e classes. É um bom ponto de partida porque:
- Treina rápido
- É fácil de interpretar (pesos/coeficientes)
- Define um "piso" de desempenho para comparação

In [7]:
model_baseline_lr = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

print("Treinando baseline (LogisticRegression)...")
model_baseline_lr.fit(X_train, y_train)
pred_baseline_lr = model_baseline_lr.predict(X_test)

res_lr = avaliar_classificacao("Baseline — LogisticRegression", y_test, pred_baseline_lr)
acc_baseline_lr = res_lr["acc"]
cm_baseline_lr = res_lr["cm"]

Treinando baseline (LogisticRegression)...

=== Baseline — LogisticRegression ===
Accuracy: 1.0000

Classification report:
              precision    recall  f1-score   support

      Adelie       1.00      1.00      1.00        30
   Chinstrap       1.00      1.00      1.00        14
      Gentoo       1.00      1.00      1.00        25

    accuracy                           1.00        69
   macro avg       1.00      1.00      1.00        69
weighted avg       1.00      1.00      1.00        69

Matriz de confusão (linhas=real, colunas=previsto):
                 Adelie   Chinstrap      Gentoo
      Adelie         30           0           0
   Chinstrap          0          14           0
      Gentoo          0           0          25


### Melhoria — Random Forest

Modelo **mais potente**: conjunto de árvores de decisão (ensemble). É considerado uma "melhoria" porque:
- Captura relações **não-lineares** entre features
- Lida melhor com interações entre variáveis
- É mais robusto a outliers

O pré-processamento é **idêntico** ao baseline — a comparação é justa.

In [8]:
model_improved_rf = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))
])

print("Treinando melhoria (RandomForestClassifier)...")
model_improved_rf.fit(X_train, y_train)
pred_improved_rf = model_improved_rf.predict(X_test)

res_rf = avaliar_classificacao("Melhoria — RandomForestClassifier", y_test, pred_improved_rf)
acc_improved_rf = res_rf["acc"]
cm_improved_rf = res_rf["cm"]

Treinando melhoria (RandomForestClassifier)...

=== Melhoria — RandomForestClassifier ===
Accuracy: 1.0000

Classification report:
              precision    recall  f1-score   support

      Adelie       1.00      1.00      1.00        30
   Chinstrap       1.00      1.00      1.00        14
      Gentoo       1.00      1.00      1.00        25

    accuracy                           1.00        69
   macro avg       1.00      1.00      1.00        69
weighted avg       1.00      1.00      1.00        69

Matriz de confusão (linhas=real, colunas=previsto):
                 Adelie   Chinstrap      Gentoo
      Adelie         30           0           0
   Chinstrap          0          14           0
      Gentoo          0           0          25


---

## Parte 5 — Comparação Final

In [9]:
print("=== Comparação Final ===")
print(f"Acurácia (LogisticRegression):      {acc_baseline_lr:.4f}")
print(f"Acurácia (RandomForestClassifier):  {acc_improved_rf:.4f}")
print("-" * 70)

labels = LABELS
for nome, cm in [("LogisticRegression", cm_baseline_lr), ("RandomForestClassifier", cm_improved_rf)]:
    total_erros = cm.sum() - cm.diagonal().sum()
    print(f"\n{nome} — erros totais: {total_erros}")
    for i, real in enumerate(labels):
        for j, pred in enumerate(labels):
            if i != j:
                print(f"  {real} → {pred}: {cm[i,j]:2d}")

=== Comparação Final ===
Acurácia (LogisticRegression):      1.0000
Acurácia (RandomForestClassifier):  1.0000
----------------------------------------------------------------------

LogisticRegression — erros totais: 0
  Adelie → Chinstrap:  0
  Adelie → Gentoo:  0
  Chinstrap → Adelie:  0
  Chinstrap → Gentoo:  0
  Gentoo → Adelie:  0
  Gentoo → Chinstrap:  0

RandomForestClassifier — erros totais: 0
  Adelie → Chinstrap:  0
  Adelie → Gentoo:  0
  Chinstrap → Adelie:  0
  Chinstrap → Gentoo:  0
  Gentoo → Adelie:  0
  Gentoo → Chinstrap:  0


## Parte 6 — Interpretação humana dos erros

### Resultado dos modelos

Ambos os modelos atingiram **100% de acurácia** no conjunto de teste, zero confusões
entre qualquer par de espécies.

### Por que zero erros?

As três espécies de pinguins são **morfologicamente muito distintas** nas features
utilizadas:

- **Gentoo** tem massa corporal e nadadeira significativamente maiores
- **Chinstrap** tem bico mais longo e fino que o Adelie
- **Adelie** tem bico mais curto e profundo

Isso significa que as fronteiras entre as classes são claras o suficiente para que
até um modelo linear (LogisticRegression) separe perfeitamente as espécies. Não
há sobreposição relevante nos dados de teste.

### O que significaria um erro neste contexto?

Se houvesse confusões, o padrão mais provável seria **Adelie ↔ Chinstrap**, pois
as duas espécies têm `bill_depth_mm` próximos (~18 mm). Uma confusão entre elas
significaria: "o modelo classificou um pinguim Adelie como Chinstrap (ou vice-versa)
baseado em medidas morfológicas ambíguas".

Confusões envolvendo **Gentoo** seriam improváveis, dado o quanto essa espécie se
diferencia das outras em massa e tamanho de nadadeira.

### Limitação do dataset

O resultado também aponta uma limitação: o `penguins` é um dataset
**relativamente simples** para classificação. Com apenas 344 amostras e classes
bem separadas, ele é mais adequado para aprender o pipeline do que para treinar
a interpretação de erros reais.